In [19]:
import asyncio
import nest_asyncio
import sys
import threading
import os
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from playwright.async_api import async_playwright
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import re

# 1. Налаштування
load_dotenv()
nest_asyncio.apply()
semaphore = asyncio.Semaphore(3)

# Конфігурація Google Sheets
SCOPE = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
CREDENTIALS_FILE = 'credentials.json'
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')

def get_sheet():
    """Авторизація та підключення до таблиці"""
    creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_FILE, SCOPE)
    client = gspread.authorize(creds)
    return client.open_by_key(SPREADSHEET_ID).sheet1

def clean_text(text, pattern=None):
    """Очищення тексту (наприклад, видалення 'грн', 'м²' тощо)"""
    if not text or text == "Не вказано":
        return ""
    if pattern:
        match = re.search(pattern, text)
        return match.group(1) if match else text
    return text.strip()

async def parse_single_ad(browser, url):
    async with semaphore: # Використовуємо ліміт одночасних вікон
        context = await browser.new_context()
        page = await context.new_page()
        await page.route("**/*.{png,jpg,jpeg,svg,webp,css,woff2}", lambda route: route.abort())
        try:
            print(f"Обробка: {url}")
            await page.goto(url, timeout=60000)
            # Чекаємо тільки завантаження структури, не чекаємо рекламу
            await page.wait_for_load_state('domcontentloaded')

            # Отримуємо HTML і віддаємо його BeautifulSoup
            html = await page.content()
            soup = BeautifulSoup(html, 'html.parser')

            # 1. Ціна (твій знайдений селектор)
            price_tag = soup.select_one('[data-testid="ad-price-container"] h3')
            price = price_tag.get_text(strip=True) if price_tag else "Не вказано"
            price = price.replace("Договірна", "").replace(" ", "")

            params_container = soup.select_one('[data-testid="ad-parameters-container"]')
            params = {}

            if params_container:
                # Знаходимо всі теги <p>, які містять двокрапку (ключ: значення)
                for p in params_container.find_all('p'):
                    text = p.get_text(strip=True)
                    if ":" in text:
                        key, val = text.split(":", 1)
                        params[key.strip()] = val.strip()

            floor = params.get("Поверх", "")
            total_floors = params.get("Поверховість", "")
            area_raw = params.get("Загальна площа", "")
            area = re.sub(r'[^\d\.]', '', area_raw.replace(",", "."))

            # 3. Населений пункт (очищення від "Продаж квартир - ")
            city_tag = soup.select('[data-testid="breadcrumb-item"]')
            city_raw = city_tag[-1].get_text(strip=True) if city_tag else "Київ"
            city = city_raw.split(' - ')[-1] # Беремо тільки останню частину назви

            return [url, price, floor, total_floors, city, area]

        except Exception as e:
            print(f"Помилка на {url}: {e}")
            return None
        finally:
            await page.close()

async def scrape_and_save():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    async with async_playwright() as p:
        print("Запуск браузера...")
        browser = await p.chromium.launch(headless=True) # Поставте True, щоб працювало у фоні
        page = await browser.new_page()

        # 2. Отримання списку оголошень
        print("Пошук оголошень на OLX...")
        await page.goto('https://www.olx.ua/uk/nedvizhimost/kvartiry/kiev/', timeout=60000)
        await page.wait_for_load_state('domcontentloaded')
        await page.wait_for_selector('a[href*="/obyavlenie/"]', timeout=60000)

        # Збираємо унікальні посилання
        locators = await page.locator('a[href*="/obyavlenie/"]').all()
        urls = list(set([await l.get_attribute('href') for l in locators]))
        urls = [u if u.startswith('http') else f"https://www.olx.ua{u}" for u in urls]

        urls_to_parse = urls[:10]
        print(f"Знайдено {len(urls)}. Починаємо асинхронний парсинг {len(urls_to_parse)} штук...")

        # Запускаємо всі парсери одночасно!
        tasks = [parse_single_ad(browser, url) for url in urls_to_parse]
        results = await asyncio.gather(*tasks)

        # Відсіюємо порожні результати (де були помилки)
        final_data = [r for r in results if r is not None]

        await browser.close()

        # 4. Запис у Google Sheets
        if final_data:
            print("Запис даних у Google Sheets...")
            sheet = get_sheet()

            first_row = sheet.row_values(1)
            headers = ["URL", "Ціна", "Поверх", "Поверховість", "Район", "Площа"]

            if first_row != headers:
                sheet.insert_row(headers, 1)

            sheet.append_rows(final_data)
            print(f"Успішно додано {len(final_data)} рядків!")
        else:
            print("Немає даних для запису.")



# Запуск через потік (як у вашому робочому прикладі)
def run_main():
    asyncio.run(scrape_and_save())

thread = threading.Thread(target=run_main)
thread.start()
thread.join()

Запуск браузера...
Пошук оголошень на OLX...
Знайдено 52. Починаємо асинхронний парсинг 10 штук...
Обробка: https://www.olx.ua/d/uk/obyavlenie/prodazh-trikmnatno-kvartiri-103m-metro-remont-vul-levka-lukyanenenka-29-ID10iaN1.html?search_reason=search%7Cpromoted
Обробка: https://www.olx.ua/d/uk/obyavlenie/akademmstechko-prodazh-2-kmnatna-pr-t-ak-palladina-24-derzhprogrami-ID10kqCH.html?search_reason=search%7Corganic
Обробка: https://www.olx.ua/d/uk/obyavlenie/dovgostrokova-orenda-suchasno-kvartiri-na-andrvskomu-uzvoz-kontraktova-ploscha-podl-vd-vlasnika-ID10fl4C.html?search_reason=search%7Corganic
Обробка: https://www.olx.ua/d/uk/obyavlenie/prodazha-kvartir-novopecherskie-lipki-bez-andreya-verhoglyada-22b-IDXL0xr.html?search_reason=search%7Cpromoted
Обробка: https://www.olx.ua/d/uk/obyavlenie/sdam-2-kmnatnu-kvartiru-v-eltnomu-zhk-varshavskiy-mkrorayon-ID10g7iw.html?search_reason=search%7Corganic
Обробка: https://www.olx.ua/d/uk/obyavlenie/2k-k-ra-67m2-varshavskiy-3-varshavskiy-bud-10-2-z